<a href="https://www.kaggle.com/code/ugokorubeast/ugoko-byu?scriptVersionId=315947157" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
from pathlib import Path
import random
import hashlib

import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ===== Phase 1 minimal training config =====
SEED = 42
NUM_TOMOS = 2
DEPTH = 16
IMG_SIZE = 96
BATCH_SIZE = 1
EPOCHS = 1
LR = 1e-3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def input_root():
    # ../input, ./input, train_data などから tomo_* ディレクトリがある場所を自動検出
    candidate_roots = [
        Path('../input'),
        Path('../input/competitions/byu-locating-bacterial-flagellar-motors-2025/train'),
        Path('./input'),
        Path('train_data'),
        Path('../train_data'),
    ]

    inner_input_root = None
    for root in candidate_roots:
        if root.exists() and any(p.is_dir() and p.name.startswith('tomo_') for p in root.iterdir()):
            inner_input_root = root
            break

    if inner_input_root is None:
        raise FileNotFoundError(
            "tomo_* ディレクトリが見つかりません。候補: ../input, ./input, train_data, ../train_data"
        )

    print(f"[INFO] input_root = {inner_input_root.resolve()}")
    return inner_input_root

[INFO] input_root = /Users/yuhaogao/workspace/BYU-competition/train_data


In [ ]:

def select_tomos(input_root):
    all_tomos = sorted([
        p for p in input_root.iterdir()
        if p.is_dir() and p.name.startswith('tomo_')
    ])

    if len(all_tomos) < NUM_TOMOS:
        raise ValueError(f"tomo フォルダ数が不足しています: found={len(all_tomos)}, required={NUM_TOMOS}")

    # seed 固定で 2 件ランダム抽出
    selected_tomos = random.sample(all_tomos, k=NUM_TOMOS)

    print('[INFO] selected_tomos:')
    for p in selected_tomos:
        print(' -', p.name)
    return selected_tomos

selected_tomos = select_tomos(input_root())

In [ ]:
processed_root = Path('./processed_tomos')
processed_root.mkdir(parents=True, exist_ok=True)


def load_tomo_volume(tomo_dir: Path, img_size: int):
    slice_paths = sorted(tomo_dir.glob('slice_*.jpg'))
    if not slice_paths:
        slice_paths = sorted(tomo_dir.glob('*.jpg'))
    if not slice_paths:
        raise FileNotFoundError(f'画像スライスが見つかりません: {tomo_dir}')

    volume = []
    for path in slice_paths:
        image = Image.open(path).convert('L').resize((img_size, img_size))
        array = np.asarray(image, dtype=np.float32) / 255.0
        volume.append(array)

    volume = np.stack(volume, axis=0)  # [D, H, W]
    return volume


# 2-1: 生データ -> .npy への前処理
# まずはこの notebook で選んだ tomo だけを処理し、必要なら all_tomos に切り替えられるようにしている。
preprocess_targets = selected_tomos

for tomo_dir in preprocess_targets:
    volume = load_tomo_volume(tomo_dir, IMG_SIZE)
    out_path = processed_root / f'{tomo_dir.name}.npy'
    np.save(out_path, volume.astype(np.float32))
    print(f'[PREPROCESS] {tomo_dir.name} -> {out_path} shape={volume.shape} dtype={volume.dtype}')

print(f'[INFO] processed_root = {processed_root.resolve()}')

In [ ]:
class CustomDataset(Dataset):
    """最小実装: 前処理済み .npy を優先し、なければ生データから [1, D, H, W] を返す。"""

    def __init__(self, tomo_dirs, depth=16, img_size=96, processed_root: Path | None = None):
        self.tomo_dirs = list(tomo_dirs)
        self.depth = depth
        self.img_size = img_size
        self.processed_root = processed_root

    def __len__(self):
        return len(self.tomo_dirs)

    def _load_slice(self, path: Path):
        img = Image.open(path).convert('L').resize((self.img_size, self.img_size))
        arr = np.asarray(img, dtype=np.float32) / 255.0
        return arr

    def _load_volume(self, tomo_dir: Path):
        if self.processed_root is not None:
            npy_path = self.processed_root / f'{tomo_dir.name}.npy'
            if npy_path.exists():
                return np.load(npy_path).astype(np.float32)

        slices = sorted(tomo_dir.glob('slice_*.jpg'))
        if not slices:
            slices = sorted(tomo_dir.glob('*.jpg'))
        if not slices:
            raise FileNotFoundError(f'画像スライスが見つかりません: {tomo_dir}')

        # depth を満たすように切り出し or 末尾リピート
        selected = slices[:self.depth]
        if len(selected) < self.depth:
            selected = selected + [selected[-1]] * (self.depth - len(selected))

        return np.stack([self._load_slice(p) for p in selected], axis=0)  # [D, H, W]

    def __getitem__(self, idx):
        tomo_dir = self.tomo_dirs[idx]
        vol = self._load_volume(tomo_dir)
        x = torch.from_numpy(vol).unsqueeze(0)  # [1, D, H, W]
        return x


train_ds = CustomDataset(
    selected_tomos,
    depth=DEPTH,
    img_size=IMG_SIZE,
    processed_root=processed_root,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

sample = next(iter(train_loader))
print('[INFO] sample shape =', tuple(sample.shape))  # [B, 1, D, H, W]

In [7]:
class Tiny3DAutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(16, 8, kernel_size=4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(8, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = Tiny3DAutoEncoder().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

print(f'[INFO] device = {device}')
print(f'[INFO] steps_per_epoch = {len(train_loader)}')

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for step, x in enumerate(train_loader, start=1):
        x = x.to(device)

        optimizer.zero_grad(set_to_none=True)
        y = model(x)
        loss = criterion(y, x)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        current_lr = optimizer.param_groups[0]['lr']
        print(f'[train] epoch={epoch} step={step}/{len(train_loader)} loss={loss.item():.6f} lr={current_lr:.2e}')

    epoch_loss = running_loss / max(1, len(train_loader))
    print(f'[epoch] epoch={epoch} avg_loss={epoch_loss:.6f}')

[INFO] device = cpu
[INFO] steps_per_epoch = 2
[train] epoch=1 step=1/2 loss=0.054690 lr=1.00e-03
[train] epoch=1 step=2/2 loss=0.042041 lr=1.00e-03
[epoch] epoch=1 avg_loss=0.048365
